# Table-7 Metric Reproduction

This notebook computes response-level hallucination rate, atomic-claim-level hallucination rate, mean end-to-end latency, fallback rate, and overall human-evaluation score from raw output/annotation files.


In [ ]:
%pip install -q pandas krippendorff


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUTS = Path("/content/table7_system_outputs.csv")
HALLUCINATION = Path("/content/hallucination_annotations.csv")
HUMAN = Path("/content/human_evaluation.csv")

assert OUTPUTS.exists(), OUTPUTS
assert HALLUCINATION.exists(), HALLUCINATION
assert HUMAN.exists(), HUMAN

outputs = pd.read_csv(OUTPUTS)
hallucination = pd.read_csv(HALLUCINATION)
human = pd.read_csv(HUMAN)

configurations = [
    "QA-only",
    "QA + KG",
    "QA + Document Retrieval",
    "QA + LLM",
    "QA + LLM + Confidence Routing",
    "Full EQAS",
]


## Metric definitions

- Response hallucination rate: `100 * count(U_i > 0) / N`
- Atomic-claim hallucination rate: `100 * sum(U_i) / sum(C_i)`
- Mean latency: `sum(latency_seconds) / N`
- Fallback rate: `100 * sum(used_fallback) / N`
- Human evaluation: arithmetic mean over all response × evaluator × dimension Likert ratings.


In [ ]:
dimensions = ["correctness","clarity","sufficiency","helpfulness"]

summary = []
for config in configurations:
    o = outputs[outputs["configuration"] == config].copy()
    h = hallucination[hallucination["configuration"] == config].copy()
    u = human[human["configuration"] == config].copy()

    assert len(o) > 0, f"No system outputs for {config}"
    assert len(h) > 0, f"No hallucination annotations for {config}"

    output_ids = set(o["id"].astype(str))
    hallucination_ids = set(h["response_id"].astype(str))
    assert output_ids == hallucination_ids, f"Output/annotation ID mismatch for {config}"

    h["hallucinated_response"] = (h["unsupported_atomic_claims"] > 0).astype(int)

    response_hr = 100.0 * h["hallucinated_response"].mean()
    claim_den = h["total_atomic_claims"].sum()
    claim_hr = (
        100.0 * h["unsupported_atomic_claims"].sum() / claim_den
        if claim_den else 0.0
    )

    mean_latency = o["latency_seconds"].astype(float).mean()
    fallback = 100.0 * o["used_fallback"].astype(bool).mean()

    human_score = np.nan
    if len(u):
        # One row represents one evaluator's ratings for one response.
        # When three evaluators are expected, every response must have 3 rows.
        evaluator_counts = u.groupby("response_id")["evaluator_id"].nunique()
        assert (evaluator_counts == 3).all(), f"Incomplete evaluator coverage for {config}"
        human_score = u[dimensions].astype(float).to_numpy().mean()

    summary.append({
        "Configuration": config,
        "Response-level Hallucination Rate (%)": response_hr,
        "Atomic-claim Hallucination Rate (%)": claim_hr,
        "Latency (s)": mean_latency,
        "Fallback Rate (%)": fallback,
        "Human Evaluation": human_score,
    })

table7 = pd.DataFrame(summary)
display(table7)
table7.to_csv("/content/table7_metrics_recomputed.csv", index=False)
